# 01 — EDA: Gold `br_inep_alfabetizacao`

Tabelas: `alunos_features`, `contexto_territorio`, `indicador_crianca_alfabetizada_municipio`, `indicador_crianca_alfabetizada_uf`  
Partições: `ano=2023`, `ano=2024` (+ `_delta_log/`).

Amostras locais: `part-*.parquet` na raiz (leitura limitada a 50 linhas).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import (
    DATALAKE_BUCKET,
    GOLD_PREFIX,
    GOLD_TABLE,
    GOLD_TABLES,
    GOLD_YEAR,
    TARGET_COL,
    gold_s3_uri,
)
from src.preprocessing import (
    inventory_local_samples,
    list_gold_objects,
    load_gold_sample,
    peek_parquet,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

print("URI:", gold_s3_uri())
print("tables:", GOLD_TABLES)
print("default table/year:", GOLD_TABLE, GOLD_YEAR)
print("target:", TARGET_COL)

## 1. Inventário dos samples locais

Mapeia cada `part-*.parquet` para a tabela Gold (por schema), sem ler o arquivo inteiro.

In [ ]:
inv = inventory_local_samples(n=50)
inv[["inferred_table", "file", "num_rows", "num_columns"]]

## 2. Amostra — `alunos_features` (tabela de ML)

Máximo **50 linhas**.

In [ ]:
df = load_gold_sample(table="alunos_features", n=50)
print(df.shape)
print(list(df.columns))
df.head()

In [ ]:
df.info()
missing = (df.isna().mean() * 100).sort_values(ascending=False).rename("pct_missing")
missing.head(25)

## 3. Target

In [ ]:
print(df[TARGET_COL].value_counts(dropna=False))
print(df[TARGET_COL].value_counts(normalize=True, dropna=False))

leakage_candidates = [
    "nivel_alfabetizacao",
    "id_aluno",
    "_ingestion_timestamp",
    "_silver_processed_at",
    "_silver_batch_id",
    "_gold_processed_at",
    "_gold_batch_id",
    "_source_table",
    "_batch_id",
    "_join_match",
]
print("leakage/id cols present:", [c for c in leakage_candidates if c in df.columns])

## 4. Distribuições e correlação (amostra)

In [ ]:
feature_candidates = [
    c for c in df.columns
    if c not in leakage_candidates + [TARGET_COL]
    and not c.startswith("_")
]
num_cols = df[feature_candidates].select_dtypes(include="number").columns.tolist()
cat_cols = df[feature_candidates].select_dtypes(exclude="number").columns.tolist()
print(f"num={len(num_cols)} cat={len(cat_cols)}")

if num_cols:
    df[num_cols[:6]].hist(bins=20, figsize=(12, 8))
    plt.tight_layout()
    plt.show()

In [ ]:
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="vlag", center=0)
    plt.title("Correlação (amostra 50 linhas — só exploratório)")
    plt.tight_layout()
    plt.show()

## 5. Outras tabelas Gold (amostra 50)

In [ ]:
for table in [
    "contexto_territorio",
    "indicador_crianca_alfabetizada_municipio",
    "indicador_crianca_alfabetizada_uf",
]:
    try:
        sample = load_gold_sample(table=table, n=50)
        print(f"\n=== {table} | shape={sample.shape} ===")
        print(list(sample.columns)[:20], "...")
        display(sample.head(2))
    except FileNotFoundError as exc:
        print(table, "->", exc)

## 6. Inventário S3 (opcional)

Requer credenciais AWS no `.env`.

In [ ]:
RUN_S3 = False  # True quando as credenciais estiverem ok

if RUN_S3:
    for table in GOLD_TABLES:
        keys = list_gold_objects(table=table, year=GOLD_YEAR)
        print(f"{table} ano={GOLD_YEAR}: {len(keys)} arquivos")
        for k in keys[:5]:
            print(" ", k)
else:
    print("S3 skip — defina RUN_S3=True após configurar o .env")

## 7. Hipóteses (preencher)

| Hipótese | Variáveis | Evidência |
|----------|-----------|-----------|
| Contexto socioeconômico influencia alfabetização | `ivs*`, `pib_per_capita` | |
| Histórico municipal (lag) é preditivo | `lag1_*` | |
| Região/rede importam | `nome_regiao`, `rede`, `amazonia_legal` | |

**Granularidade:** aluno (`alunos_features`)  
**Target:** `alfabetizado`  
**Excluir do modelo:** ver lista `leakage_candidates` acima